# Module 5 — Inverse Kinematics
## Unit 1 — The Inverse Problem
### Lesson 1.2 — Why It's Hard: Nonlinear, and 0/1/Many Solutions

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
Count solutions by where the target sits in the reach annulus.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    """Forward kinematics: planar 2-link gripper position."""
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def reachable(x, y, L1, L2, tol=1e-9):
    r = np.hypot(x, y)
    return abs(L1-L2)-tol <= r <= L1+L2+tol

def ik_two_link(x, y, L1, L2):
    """Return both (theta1,theta2) solutions; [] if unreachable, one if on boundary."""
    r2 = x*x + y*y
    c2 = (r2 - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        t2 = sign*np.arccos(c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1+L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(np.sin(t2)) < 1e-6:    # boundary: the two coincide
            break
    return sols

L1=L2=0.3
def count_solutions(x,y,L1,L2,tol=1e-9):
    r=np.hypot(x,y)
    if r> L1+L2+tol or r< abs(L1-L2)-tol: return 0
    if abs(r-(L1+L2))<tol or abs(r-abs(L1-L2))<tol: return 1
    return 2
for tgt in [(0.7,0),(0.6,0),(0.3,0.2)]:
    print(tgt, "->", count_solutions(*tgt,L1,L2,tol=1e-6), "solution(s)")


## Coding Exercise (§8)
Implement and test count_solutions on the three worked-example targets.


In [ ]:
# YOUR CODE HERE
assert count_solutions(0.7,0,L1,L2,tol=1e-6)==0
assert count_solutions(0.6,0,L1,L2,tol=1e-6)==1
assert count_solutions(0.3,0.2,L1,L2,tol=1e-6)==2
print("counts OK")


## Check your work


In [ ]:
# two distinct solutions strictly inside; verify both land on target via FK
sols=ik_two_link(0.3,0.2,L1,L2)
assert len(sols)==2
for (t1,t2) in sols:
    assert np.allclose(fk_two_link(t1,t2,L1,L2),[0.3,0.2],atol=1e-9)
print("All checks passed.")
